#36615 - Final Project Part 2 - Feature Engineering
## Team Polyhymnia
### Vraj Patel, Jay Louissant, Elaine Yin

#### Description:

This part aims to generate additional features for each observation to support the development of a predictive model for flight delays.

In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Window
from pyspark.sql.types import DoubleType, TimestampType


In [0]:
df = spark.table("lsd_2026.default.polyhymnia_cleaned_airline")


In [0]:
# convert FL_DATE to unix timestamp
df = df.withColumn("date_seconds", F.unix_timestamp("FL_DATE"))

# scheduled departure timestamp
df = df.withColumn("crs_dep_timestamp", 
                   (F.col("date_seconds") + F.col("CRS_DEP_TIME") * 60).cast(TimestampType()))

# scheduled arrival timestamp (+1 day if crossing midnight)
df = df.withColumn("crs_arr_timestamp", 
    F.when(
        F.col("CRS_ARR_TIME") < F.col("CRS_DEP_TIME"), 
        (F.col("date_seconds") + F.col("CRS_ARR_TIME") * 60 + 86400).cast(TimestampType()) # +1 day
    ).otherwise(
        (F.col("date_seconds") + F.col("CRS_ARR_TIME") * 60).cast(TimestampType()) # same day
    )
)



#### Feature 1 - Day of the Week

In [0]:
# derive a day of the week column from the date column
df = df.withColumn(
    "day_of_week",
    F.dayofweek("FL_DATE")  # 1=Sunday, ..., 7=Saturday
)

# see day of the week distribution for flights in the dataset
df.groupBy("day_of_week").count().orderBy("day_of_week").show()

#### Feature 2 - Rate of weather delays at departure airport in previous hour

In [0]:
df = df.withColumn("hour_of_day", F.hour(F.col("crs_dep_timestamp")))

# Define the rolling window: based on the scheduled time (CRS), consider the previous 1 hour.
window_origin_1h = Window.partitionBy("ORIGIN").orderBy(F.col("crs_dep_timestamp").cast("long")).rangeBetween(-3600, -1)
window_dest_1h = Window.partitionBy("DEST").orderBy(F.col("crs_arr_timestamp").cast("long")).rangeBetween(-3600, -1)


In [0]:
# Define Departure Weather Delay
# If either value is Null, the condition evaluates to False -> 0.0
df = df.withColumn("is_origin_weather_delay", 
                   F.when(
                       (F.col("WEATHER_DELAY") > 0) & (F.col("DEP_DELAY") > 0), 
                       1.0
                   ).otherwise(0.0))

# Define Arrival Weather Delay
df = df.withColumn("is_dest_weather_delay", 
                   F.when(
                       (F.col("WEATHER_DELAY") > 0) & 
                       (F.col("DEP_DELAY") <= 0) & 
                       (F.col("ARR_DELAY") > 0), 
                       1.0
                   ).otherwise(0.0))



In [0]:
# Feature 2: 1-hour departure weather delay rate

# Numerator: number of flights that meet the "Departure Weather Delay" condition
df = df.withColumn("dep_weather_sum", F.sum("is_origin_weather_delay").over(window_origin_1h))

# Denominator: total number of flights scheduled to depart within the time window
df = df.withColumn("dep_flight_count_1h", F.count("*").over(window_origin_1h))

# Compute the ratio 
df = df.withColumn("rate_weather_delay_dep", 
                   F.when(F.col("dep_flight_count_1h") > 0, 
                          F.coalesce(F.col("dep_weather_sum"), F.lit(0.0)) / F.col("dep_flight_count_1h"))
                   .otherwise(0.0))


#### Feature 3 - Rate of weather delays from the arrival airport in the previous hour

In [0]:
# Feature 3: Previous 1-hour arrival weather delay rate
df = df.withColumn("arr_weather_sum", F.sum("is_dest_weather_delay").over(window_dest_1h))
df = df.withColumn("arr_flight_count_1h", F.count("*").over(window_dest_1h))

df = df.withColumn("rate_weather_delay_arr", 
                   F.when(F.col("arr_flight_count_1h") > 0, 
                          F.coalesce(F.col("arr_weather_sum"), F.lit(0.0)) / F.col("arr_flight_count_1h"))
                   .otherwise(0.0))

#### Feature 4 - Number of flights departing from the departure airport in the previous hour, compared the average number during this hour on the same day of the week, as a z score

In [0]:
# Feature 4: Departure traffic z-score 
stats_df = df.groupBy("ORIGIN", "day_of_week", "hour_of_day") \
             .agg(F.mean("dep_flight_count_1h").alias("avg_flights_hist"),
                  F.stddev("dep_flight_count_1h").alias("std_flights_hist"))

df = df.join(stats_df, on=["ORIGIN", "day_of_week", "hour_of_day"], how="left")

# Compute z-score; default to 0 when std is NULL or zero
df = df.withColumn("dep_traffic_z_score", 
                   F.when((F.col("std_flights_hist").isNotNull()) & (F.col("std_flights_hist") != 0),
                          (F.col("dep_flight_count_1h") - F.col("avg_flights_hist")) / F.col("std_flights_hist"))
                   .otherwise(0.0))

In [0]:
# Verify additional features created properly
df.select(
    "day_of_week", 
    "hour_of_day", 
    "rate_weather_delay_dep", 
    "rate_weather_delay_arr", 
    "dep_traffic_z_score"
).show(10, truncate=False)


#### Additional Feature 1 - Carrier-specific historical delay rate

The carrier-specific historical delay rate captures the fraction of flights delayed by the same OP_CARRIER over a trailing window of recent time, allowing the model to incorporate short-term operational performance into its predictions. Some airlines are better (or worse) at staying on schedule, which may reflect differences in organizational structure, fleet management strategies, staffing levels, maintenance practices, and scheduling policies. By computing this feature over a rolling time window rather than across the entire dataset, we allow the model to capture dynamic changes in performance, such as temporary disruptions, staffing shortages, weather spillover effects, or operational improvements. This feature therefore provides a meaningful summary of an airline’s recent reliability and helps the model distinguish between systemic carrier-level performance patterns and one-off flight-specific factors.

In [0]:
# Helper: define the binary label "is_delayed"

df = df.withColumn(
    "is_delayed",
    F.when((F.col("ARR_DELAY") > 0) | (F.col("CANCELLED") == 1), F.lit(1)).otherwise(F.lit(0))
)

In [0]:
seconds_in_day = 86400
trail_days = 7
trail_seconds = trail_days * seconds_in_day

carrier_window_7d = (
    Window
    .partitionBy("OP_CARRIER")
    .orderBy(F.col("crs_dep_timestamp").cast("long"))
    .rangeBetween(-trail_seconds, -1)  
    # past 7 days, exclude current flight
)

# avg of 0/1 = fraction delayed
df = df.withColumn(
    "carrier_delay_rate_7d",
    F.avg("is_delayed").over(carrier_window_7d)  
)

# Fill nulls (first flights for a carrier have no "past 7 days")
df = df.withColumn(
    "carrier_delay_rate_7d",
    F.coalesce(F.col("carrier_delay_rate_7d"), F.lit(0.0))
)


#### Additional Feature 2 - Route-level historical delay rate


The route-level historical delay rate measures the fraction of flights delayed for a specific (ORIGIN, DEST) pair over a recent time window. Some routes are simply more prone to delays due to busy airspace, congestion, or recurring weather patterns. While WEATHER_DELAY captures delays officially attributed to weather for that specific flight, it only reflects direct causes. Weather can also create indirect or cascading effects, such as earlier storms delaying incoming aircraft, regional airspace disruptions, or congestion spilling over from nearby cities, which may be coded under different delay types. This rolling route-level delay rate helps capture those broader, persistent patterns and provides useful context about how reliable a route typically is.

In [0]:
route_window_7d = (
    Window
    .partitionBy("ORIGIN", "DEST")
    .orderBy(F.col("crs_dep_timestamp").cast("long"))
    .rangeBetween(-trail_seconds, -1)
)

df = df.withColumn(
    "route_delay_rate_7d",
    F.avg("is_delayed").over(route_window_7d)
)

df = df.withColumn(
    "route_delay_rate_7d",
    F.coalesce(F.col("route_delay_rate_7d"), F.lit(0.0))
)

#### Additional Feature 3 - Change CRS_DEP_TIME and ARR_TIME into buckets


Binning CRS_DEP_TIME and ARR_TIME into time-of-day buckets allows the model to capture systematic patterns in delays that vary throughout the day. Delay behavior differs dramatically by time of day. For example, early morning flights tend to be more reliable, while later flights often inherit delays from earlier disruptions. By keeping the original continuous time variable and also creating categorical time buckets, we allow the model to learn both fine-grained timing effects and broader structural patterns, such as peak congestion periods. In this case, binning can actually add useful information rather than lose it, since it highlights meaningful shifts in operational dynamics across different parts of the day.

In [0]:

def time_bucket(col_minutes):
    """
    Create a simple time-of-day bucket from minutes since midnight.
    Buckets are coarse on purpose (good for beginner/intermediate modeling).
    """
    return (
        F.when(col_minutes.isNull(), F.lit("unknown"))
         # 00:00 - 04:59
         .when((col_minutes >= 0)   & (col_minutes < 300),  F.lit("late_night"))
         # 05:00 - 07:59
         .when((col_minutes >= 300) & (col_minutes < 480),  F.lit("early_morning"))
         # 08:00 - 10:59
         .when((col_minutes >= 480) & (col_minutes < 660),  F.lit("morning"))
         # 11:00 - 13:59
         .when((col_minutes >= 660) & (col_minutes < 840),  F.lit("midday"))
         # 14:00 - 16:59
         .when((col_minutes >= 840) & (col_minutes < 1020), F.lit("afternoon"))
         # 17:00 - 20:59
         .when((col_minutes >= 1020)& (col_minutes < 1260), F.lit("evening"))
         # 21:00 - 23:59
         .when((col_minutes >= 1260)& (col_minutes < 1440), F.lit("night"))
         .otherwise(F.lit("unknown"))
    )

df = df.withColumn("crs_dep_time_bucket", time_bucket(F.col("CRS_DEP_TIME")))
df = df.withColumn("arr_time_bucket",     time_bucket(F.col("ARR_TIME")))

In [0]:
# verify additional features created properly
df.select(
    "OP_CARRIER", "ORIGIN", "DEST", "CRS_DEP_TIME", "ARR_TIME",
    "crs_dep_time_bucket", "arr_time_bucket",
    "carrier_delay_rate_7d", "route_delay_rate_7d"
).show(10, truncate=False)

In [0]:
# save dataframe as table
table_name = "lsd_2026.default.polyhymnia_feature_engineered"
df.write.saveAsTable(table_name)